# Face_ID_v1 - Expérimentations et Évaluation
## Reconnaissance faciale intelligente des étudiants

Ce notebook documente la chaîne complète de traitement du projet académique **Face_ID_v1** :
1. **Détection faciale** via YuNet (Deep Learning).
2. **Extraction des représentations numériques** (embeddings 128D) via SFace.
3. **Identification des étudiants** et calcul des similarités cosinus.
4. **Gestion des personnes inconnues** et impact du seuil de décision.
5. **Évaluation quantitative** et génération du fichier `results/test_results.csv`.

In [ ]:
import sys
from pathlib import Path

# Ajout du dossier racine au sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.detect_faces import FaceDetector
from src.encode_faces import FaceEncoder, encode_dataset
from src.recognize_faces import FaceRecognizer
from src.evaluate import evaluate_dataset

print(f"OpenCV version : {cv2.__version__}")
print(f"NumPy version : {np.__version__}")

## 1. Initialisation des modèles de Deep Learning (YuNet + SFace)

Nous utilisons deux modèles pré-entraînés issus de l'OpenCV Model Zoo :
- **YuNet** : détecteur de visages léger et performant avec extraction de 5 repères faciaux (landmarks).
- **SFace** : réseau de neurones profonds générant un vecteur d'empreinte faciale de dimension 128.

In [ ]:
detector = FaceDetector(conf_threshold=0.7)
encoder = FaceEncoder(detector=detector)
recognizer = FaceRecognizer(cosine_threshold=0.363, encoder=encoder, detector=detector)

print("Modèles initialisés avec succès !")
print(f"Nombre d'étudiants actuellement en base : {len(set(recognizer.known_names))}")

## 2. Encodage du jeu de données (`dataset/`)

Cette étape parcourt les répertoires `dataset/student_01` à `student_05`, extrait les vecteurs d'empreintes faciales et les sauvegarde dans `models/encodings.pkl`.

In [ ]:
data_encodings = encode_dataset()
recognizer.reload_encodings()
print(f"Base rechargée : {len(recognizer.known_encodings)} visages enregistrés.")

## 3. Analyse de la métrique de similarité cosinus

La similarité cosinus entre deux vecteurs $\mathbf{u}$ et $\mathbf{v}$ est donnée par :
$$\text{CosSim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

- Si $\text{CosSim} \ge \text{seuil}$, la prédiction est acceptée.
- Si $\text{CosSim} < \text{seuil}$, l'individu est classé comme **« Inconnu »**.

In [ ]:
# Visualisation de la distribution des scores si des données sont disponibles
if len(recognizer.known_encodings) >= 2:
    scores = []
    for i in range(len(recognizer.known_encodings)):
        for j in range(i + 1, len(recognizer.known_encodings)):
            s = encoder.compute_similarity(recognizer.known_encodings[i], recognizer.known_encodings[j])
            scores.append(s)
    
    plt.figure(figsize=(8, 4))
    plt.hist(scores, bins=20, color='skyblue', edgecolor='black')
    plt.axvline(0.363, color='red', linestyle='--', label='Seuil recommandé (0.363)')
    plt.title('Distribution des similarités cosinus inter-visages')
    plt.xlabel('Similarité cosinus')
    plt.ylabel('Effectif')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 4. Évaluation quantitative et export des résultats

Exécution des tests et génération du tableau récapitulatif `results/test_results.csv` conformément à la section 10 du README.

In [ ]:
df_results = evaluate_dataset(recognizer, PROJECT_ROOT / "dataset", PROJECT_ROOT / "results" / "test_results.csv")
if not df_results.empty:
    display(df_results.head(10))
else:
    print("Aucune donnée dans le dataset pour le moment.")